# Combining New Variables (Extracted August 2025)
**Prepared by AIF, 2025-08-14**
Converted to Python/Pandas from the original R script.

### 1. Setup, Load Libraries & Define Directories

In [ ]:
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv(override=True)
env_data_dir = os.getenv("DATA_DIR")
if env_data_dir is None:
	raise ValueError("DATA_DIR environment variable is not set. Please set it in the .env file.")

pd.options.mode.chained_assignment = None  # default='warn'

# define paths
user = "lazycst"
root_dir = Path(os.getcwd()).parent.parent

work_dir = root_dir / "build/src/"
data_dir = Path(env_data_dir)						# On mobile env, set DATA_DIR in .env to "H:/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30"
output_dir = root_dir / "build/output/"

output_dir.mkdir(parents=True, exist_ok=True)

# list industry subfolders
industries = [d.name for d in data_dir.iterdir() if d.is_dir()]
print(f"Found {len(industries)} industry folders.")
print(industries)

Found 30 industry folders.
['01_12', '100', '13_20', '21', '22_30', '31', '32_36', '37_39', '41', '42', '43', '45_46', '47', '49_52_53_55', '50_51', '56', '58_59_60_61_63', '62', '64_65_66_69', '68', '70', '71_74', '75_81', '82', '84_86', '87_90', '91_95', '96', '97_99', 'output']


### Load the CLEAN dataset
Read the master panel Excel file to get the IDs we'll merge with.

In [ ]:
from janitor import clean_names

clean_db = data_dir / "output/master_panel.xlsx"
print(f"Reading clean dataset from: {clean_db}")

try:

	# Standardizes column names (e.g., lowercases them, replaces spaces with underscores) 
    # to avoid typos when referencing them later.
    clean_current = pd.read_excel(clean_db)
    clean_current = clean_names(clean_current)

	# Forces these two columns to be strings to ensure consistency. 
    # This prevents issues where '12345' (string) and 12345 (integer) are treated differently.
    clean_current['registered_number'] = clean_current['registered_number'].astype(str)
    clean_current['company_name'] = clean_current['company_name'].astype(str)
    
    # Create merge_id: coalesce registered_number (if not empty/nan) else company_name
    clean_current['merge_id'] = clean_current['registered_number'].replace(['', 'nan', 'None'], np.nan)
    clean_current['merge_id'] = clean_current['merge_id'].fillna(clean_current['company_name'])
    
    keep_ids = clean_current[['merge_id']].dropna().drop_duplicates()
    keep_ids = keep_ids[keep_ids['merge_id'] != ""]

	# Checks if the 'year' column exists in this dataset, returning True or False.
    # This acts as a flag to tell the program later whether it needs to merge by just 'merge_id' or
	# by ['merge_id', 'year']    
    has_year_in_clean = 'year' in clean_current.columns
    print(f"Clean dataset loaded. has_year_in_clean: {has_year_in_clean}")
    
except Exception as e:
    print(f"Error reading clean data: {e}. \n(Make sure the path exists on this machine)")
    keep_ids = None
    has_year_in_clean = False

SAVE_PER_INDUSTRY = False

Reading clean dataset from: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.07.30\output\master_panel.xlsx
Error reading clean data: [Errno 2] No such file or directory: 'D:\\FAME - LN dataset\\Dropbox\\fame_clean\\1_FAME_raw_data\\2025.07.30\\output\\master_panel.xlsx'. 
(Make sure the path exists on this machine)


### 2. Helper Functions

In [ ]:
def parse_excel_or_iso_date(s):
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    if not s:
        return pd.NaT
    # Try numeric excel date first
    try:
        num = float(s)
        return pd.to_datetime(num, unit='D', origin='1899-12-30')
    except ValueError:
        pass
    # Try standard string parse (infer_datetime_format)
    try:
        return pd.to_datetime(s, dayfirst=True)
    except Exception:
        return pd.NaT

def parse_accounting_reference_md_str(s):
    d = parse_excel_or_iso_date(s)
    if pd.isna(d):
        # check if it's just a dd/mm or mm/dd format (like '31/12')
        s_str = str(s).strip()
        if re.match(r'^\d{1,2}/\d{1,2}$', s_str):
            try:
                d_test = pd.to_datetime(s_str + '/2000', dayfirst=True)
                return d_test.strftime('%m-%d')
            except Exception:
                pass
        return np.nan
    return d.strftime('%m-%d')

def fix_date_vars(df):
    if 'status_date' in df.columns:
        df['status_date'] = df['status_date'].apply(parse_excel_or_iso_date)
    if 'date_of_incorporation' in df.columns:
        df['date_of_incorporation'] = df['date_of_incorporation'].apply(parse_excel_or_iso_date)
    if 'accounting_reference_date' in df.columns:
        df['accounting_reference_date'] = df['accounting_reference_date'].apply(parse_accounting_reference_md_str)
    return df

### 3. Main loop across subfolders and industries

In [5]:
# import help functions from './helpers.py'
from janitor import clean_names
from helpers import parse_excel_or_iso_date, parse_accounting_reference_md_str, fix_date_vars

years = [str(y) for y in range(2006, 2024)]

fixed_vars = ["company_name", "registered_number", "inactive", "quoted", "own_data", "woco",
              "bv_d_id_number", "company_status", "status_date", "legal_form", "date_of_incorporation",
              "accounting_reference_date", "registered_accounts_type", "jordans_company_classification", 
              "account_currency", "guo_name", "guo_bv_d_id_number", "duo_name", "duo_bv_d_id_number"]

yearly_vars = [
    "cost_of_sales_th_gbp",
    "exceptional_items_pre_gp_th_gbp",
    "cash_out_in_flow_investing_activ_th_gbp",
    "capital_expenditure_financ_invest_th_gbp",
    "acquisition_disposal_th_gbp",
    "equity_dividends_paid_th_gbp"
]

key_vars = fixed_vars + ['year'] if has_year_in_clean else fixed_vars

all_industries_panels = []

for industry in industries:
    print(f"\n=== Processing industry: {industry} ===")
    folder_path = data_dir / industry
    excel_files = list(folder_path.glob("*.xlsx"))
    
    if not excel_files:
        print("  (no Excel files found)")
        continue
        
    lst = []
    for file_path in excel_files:
        try:
            # Ensure we're reading as string to emulate col_types="text"
            df = pd.read_excel(file_path, sheet_name="Results", dtype=str)
            
            # Clean column names similar to janitor (lowercase, replace spaces/special chars)
            df.columns = [re.sub(r'\.\.\d+$', '', col) for col in df.columns]
            df.columns = [re.sub(r'\.$', '', col) for col in df.columns]
            df = clean_names(df)
            
            # Handle duplicates across columns
            df = df.loc[:, ~df.columns.duplicated()]
            lst.append(df)
        except Exception as e:
            print(f"  ! Skipping (can't read 'Results'): {file_path.name}")
            
    if not lst:
        print("  (no readable 'Results' sheets)")
        continue
        
    # Row-bind and remove duplicates
    merged_data = pd.concat(lst, ignore_index=True).drop_duplicates()
    
    # Early filter
    if keep_ids is not None:
        merged_data['merge_id'] = merged_data['registered_number'].replace(['', 'nan', 'None'], np.nan)
        merged_data['merge_id'] = merged_data['merge_id'].fillna(merged_data['company_name'])
        merged_data = merged_data[merged_data['merge_id'].isin(keep_ids['merge_id'])]
        
    # Safe names cleanup
    merged_data.columns = [re.sub(r'\.x(\.\.\d+)?$', '', col) for col in merged_data.columns]
    
    # Pivot wide to long
    melted_dfs = []
    for varname in yearly_vars:
        # Find columns that start with varname and end with a year
        regex = f"^{varname}_?(\\d{{4}})$"
        cols_to_melt = [c for c in merged_data.columns if re.match(regex, c)]
        cols_to_keep = [c for c in fixed_vars if c in merged_data.columns]
        
        if not cols_to_melt:
            continue
            
        subset = merged_data[cols_to_keep + cols_to_melt]
        long_df = subset.melt(id_vars=cols_to_keep, value_vars=cols_to_melt, var_name="raw_col", value_name=varname)
        
        # Extract year from the column name
        long_df['year'] = long_df['raw_col'].str.extract(r'(\d{4})')[0]
        long_df = long_df.drop(columns=['raw_col'])
        long_df = long_df[long_df['year'].isin(years)]
        melted_dfs.append(long_df)
        
    # Merge all melted subsets together
    if melted_dfs:
        panel = melted_dfs[0]
        for i in range(1, len(melted_dfs)):
            join_cols = [c for c in fixed_vars + ['year'] if c in panel.columns and c in melted_dfs[i].columns]
            panel = pd.merge(panel, melted_dfs[i], on=join_cols, how='left')
    else:
        # No yearly vars found, use basic frame
        panel = merged_data
        
    # Numeric conversion for yearly vars
    for v in yearly_vars:
        if v in panel.columns:
            panel[v] = pd.to_numeric(panel[v], errors='coerce')
            
    # Fix dates
    panel = fix_date_vars(panel)
    
    # De-dupe
    existing_key_vars = [k for k in key_vars if k in panel.columns]
    panel = panel.drop_duplicates(subset=existing_key_vars)
    
    panel['industry_folder'] = industry
    all_industries_panels.append(panel)
    
    if SAVE_PER_INDUSTRY:
        panel.to_excel(output_dir / f"industry_{industry}.xlsx", index=False)

print("\n✅ Finished per-industry processing (objects kept in memory).")


=== Processing industry: 100 ===
  ! Skipping (can't read 'Results'): Export 14_08_2025 14_49.xlsx
  (no readable 'Results' sheets)

=== Processing industry: 21 ===
  ! Skipping (can't read 'Results'): Export 09_08_2025 10_16.xlsx
  (no readable 'Results' sheets)

=== Processing industry: 97_99 ===
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_24.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_24 1.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_24 2.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_24 3.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_24 4.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_25.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_25 1.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_25 2.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_25 4.xlsx
  ! Skipping (can't read 'Results'): Export 08_08_2025 17_25 5.xlsx
  ! Skipping (can't rea

KeyboardInterrupt: 

### 5. Merge with the clean dataset and SAVE ONLY the smaller result

In [ ]:
if all_industries_panels:
    master_panel = pd.concat(all_industries_panels, ignore_index=True)
    existing_key_vars = [k for k in key_vars if k in master_panel.columns]
    master_panel = master_panel.drop_duplicates(subset=existing_key_vars)
    
    print(f"Industries processed: {len(all_industries_panels)}")
    print(f"Rows in master_panel: {len(master_panel)}")

    # Build comparable keys
    master_panel['merge_id'] = master_panel['registered_number'].replace(['', 'nan', 'None'], np.nan)
    master_panel['merge_id'] = master_panel['merge_id'].fillna(master_panel['company_name'])
    
    mp_firms = master_panel[['merge_id']].drop_duplicates()
    
    try:
        clean_firms = clean_current[['merge_id', 'company_name']].drop_duplicates()
        matched_firms = pd.merge(clean_firms, mp_firms, on='merge_id')
        
        unmatched_firms = clean_firms[~clean_firms['merge_id'].isin(mp_firms['merge_id'])]
        
        print("\nMerge diagnostics - firm level:")
        print(f"  Matched firms: {len(matched_firms)}")
        print(f"  Unmatched firms: {len(unmatched_firms)}")
        
        not_merged_path = output_dir / "firms_not_merged_unique.xlsx"
        unmatched_firms.to_excel(not_merged_path, index=False)
        print(f"📄 Wrote unique firms not merged to: {not_merged_path}")
        
        if has_year_in_clean:
            master_panel['year'] = master_panel['year'].astype(int)
            clean_current['year'] = clean_current['year'].astype(int)
            
            final_merged = pd.merge(
                clean_current,
                master_panel.drop(columns=['registered_number', 'company_name']),
                on=['merge_id', 'year'],
                how='left'
            )
        else:
            final_merged = pd.merge(
                clean_current,
                master_panel.drop(columns=['registered_number', 'company_name']),
                on='merge_id',
                how='left'
            )
        
        final_merged = final_merged.drop(columns=['merge_id'])
        
        final_output = output_dir / "merged_all_variables_2025_08_27.xlsx"
        final_merged.to_excel(final_output, index=False)
        print(f"\n📦 Saved merged result only: {final_output}")
        print(f"Rows clean_current vs final_merged: {len(clean_current)} vs {len(final_merged)}")
        
    except NameError:
        print("\nclean_current wasn't loaded successfully, skipping merge section.")